# Hyperparameter Search and Overfitting Analysis

This notebook demonstrates the enhanced model training capabilities with:
- Train/validation/test splits
- Random hyperparameter search
- Overfitting detection and analysis
- Performance visualization

## Overview

The `EnhancedModelTrainer` class provides:
1. **Proper data splits**: Train (60%), Validation (20%), Test (20%)
2. **Random hyperparameter search**: Efficiently explores parameter space
3. **Overfitting detection**: Monitors training vs validation performance
4. **Early stopping**: Prevents wasteful computation
5. **Comprehensive analysis**: Detailed reports and visualizations

In [ ]:
import sys
import os
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from surprise import Dataset

from src.data_processing import load_letterboxd_data, create_surprise_dataset
from src.model_training import EnhancedModelTrainer

# Configure plotting
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)

print("📚 Libraries imported successfully!")

## 1. Load and Prepare Data

In [ ]:
# Load data
data_path = "../alex_data"

if os.path.exists(data_path):
    print("Loading Letterboxd data...")
    data = load_letterboxd_data(data_path)
    surprise_data = create_surprise_dataset(data)
    print(f"✅ Loaded {len(data)} ratings from Letterboxd data")
else:
    print("Using MovieLens dataset for demonstration...")
    surprise_data = Dataset.load_builtin('ml-100k')
    print("✅ Loaded MovieLens 100k dataset")

# For demo purposes, use a smaller subset for faster execution
print("\n🔬 Using smaller dataset for demonstration...")
raw_data = surprise_data.raw_ratings[:5000]  # Use first 5000 ratings
surprise_data = Dataset.load_from_df(
    pd.DataFrame(raw_data, columns=['userID', 'itemID', 'rating']), 
    surprise_data.reader
)
print(f"📊 Dataset size: {len(raw_data)} ratings")

## 2. Initialize Enhanced Model Trainer

In [ ]:
# Initialize enhanced trainer with train/validation/test splits
trainer = EnhancedModelTrainer(
    surprise_data, 
    val_size=0.2,    # 20% for validation
    test_size=0.2,   # 20% for testing
    random_state=42  # For reproducibility
)

print("✅ Enhanced model trainer initialized!")
print(f"📈 Training set: {trainer.trainset.n_ratings} ratings")
print(f"📊 Validation set: {len(trainer.validset)} ratings")
print(f"🧪 Test set: {len(trainer.testset)} ratings")

## 3. Run Hyperparameter Search

We'll perform random hyperparameter search with overfitting detection:

In [ ]:
# Run hyperparameter search for both KNN and SVD models
print("🔍 Starting hyperparameter search...")
print("This will test random combinations of hyperparameters and detect overfitting.\n")

search_results = trainer.random_hyperparameter_search(
    model_type='both',  # Search both KNN and SVD models
    n_iter=20,          # Try 20 random combinations per model
    patience=5,         # Early stopping after 5 iterations without improvement
    save_results=True   # Save detailed results
)

print("\n✅ Hyperparameter search completed!")

## 4. Analyze Search Results

In [ ]:
# Display best hyperparameters for each model
print("🏆 BEST HYPERPARAMETERS FOUND:")
print("=" * 50)

for model_name, results in search_results.items():
    if results['best_params'] is not None:
        print(f"\n{model_name}:")
        print(f"  📊 Best Validation RMSE: {results['best_val_rmse']:.4f}")
        print(f"  ⚙️  Best Parameters: {results['best_params']}")
        print(f"  🔄 Total Iterations: {results['total_iterations']}")
    else:
        print(f"\n{model_name}: No valid results found")

## 5. Visualize Hyperparameter Search Progress

In [ ]:
# Plot hyperparameter search results
trainer.plot_hyperparameter_search_results()
plt.suptitle('Hyperparameter Search Progress and Overfitting Analysis', fontsize=16, y=0.98)
plt.show()

## 6. Generate Overfitting Analysis Report

In [ ]:
# Generate comprehensive overfitting report
overfitting_report = trainer.generate_overfitting_report()

print("🔍 OVERFITTING ANALYSIS REPORT:")
print("=" * 50)
print(overfitting_report.to_string(index=False))

# Create visualization of overfitting scores
if not overfitting_report.empty:
    plt.figure(figsize=(10, 6))
    
    # Bar plot of overfitting scores
    plt.subplot(1, 2, 1)
    colors = ['green' if score < 0.05 else 'orange' if score < 0.1 else 'red' 
              for score in overfitting_report['Best_Overfitting_Score']]
    plt.bar(overfitting_report['Model'], overfitting_report['Best_Overfitting_Score'], color=colors)
    plt.title('Overfitting Scores by Model')
    plt.ylabel('Overfitting Score (Val - Train RMSE)')
    plt.xticks(rotation=45)
    plt.axhline(y=0.05, color='orange', linestyle='--', alpha=0.7, label='Moderate threshold')
    plt.axhline(y=0.1, color='red', linestyle='--', alpha=0.7, label='High threshold')
    plt.legend()
    
    # Performance vs overfitting scatter
    plt.subplot(1, 2, 2)
    plt.scatter(overfitting_report['Best_Val_RMSE'], overfitting_report['Best_Overfitting_Score'], 
                c=colors, s=100, alpha=0.7)
    for i, model in enumerate(overfitting_report['Model']):
        plt.annotate(model, 
                    (overfitting_report.iloc[i]['Best_Val_RMSE'], 
                     overfitting_report.iloc[i]['Best_Overfitting_Score']),
                    xytext=(5, 5), textcoords='offset points', fontsize=9)
    plt.xlabel('Validation RMSE')
    plt.ylabel('Overfitting Score')
    plt.title('Performance vs Overfitting Trade-off')
    plt.axhline(y=0.05, color='orange', linestyle='--', alpha=0.7)
    plt.axhline(y=0.1, color='red', linestyle='--', alpha=0.7)
    
    plt.tight_layout()
    plt.show()

## 7. Final Model Evaluation on Test Set

In [ ]:
# Evaluate final models on the test set
final_results = trainer.evaluate_final_models()

print("🧪 FINAL TEST SET EVALUATION:")
print("=" * 50)

# Create comparison DataFrame
comparison_data = []
for model_name, results in final_results.items():
    comparison_data.append({
        'Model': model_name,
        'Train_RMSE': f"{results['train_rmse']:.4f}",
        'Val_RMSE': f"{results['val_rmse']:.4f}",
        'Test_RMSE': f"{results['test_rmse']:.4f}",
        'Generalization_Gap': f"{results['generalization_gap']:.4f}",
        'Test/Train_Ratio': f"{results['test_to_train_ratio']:.3f}"
    })

comparison_df = pd.DataFrame(comparison_data)
print(comparison_df.to_string(index=False))

# Visualize final results
plt.figure(figsize=(12, 8))

# Performance comparison
plt.subplot(2, 2, 1)
models = list(final_results.keys())
train_scores = [final_results[m]['train_rmse'] for m in models]
val_scores = [final_results[m]['val_rmse'] for m in models]
test_scores = [final_results[m]['test_rmse'] for m in models]

x = np.arange(len(models))
width = 0.25

plt.bar(x - width, train_scores, width, label='Train', alpha=0.8)
plt.bar(x, val_scores, width, label='Validation', alpha=0.8)
plt.bar(x + width, test_scores, width, label='Test', alpha=0.8)

plt.xlabel('Models')
plt.ylabel('RMSE')
plt.title('Performance Comparison Across Data Splits')
plt.xticks(x, models, rotation=45)
plt.legend()
plt.grid(True, alpha=0.3)

# Generalization gap
plt.subplot(2, 2, 2)
gaps = [final_results[m]['generalization_gap'] for m in models]
colors = ['green' if gap < 0.05 else 'orange' if gap < 0.1 else 'red' for gap in gaps]
plt.bar(models, gaps, color=colors, alpha=0.7)
plt.xlabel('Models')
plt.ylabel('Generalization Gap (Test - Train RMSE)')
plt.title('Generalization Gap Analysis')
plt.xticks(rotation=45)
plt.axhline(y=0.05, color='orange', linestyle='--', alpha=0.7)
plt.axhline(y=0.1, color='red', linestyle='--', alpha=0.7)
plt.grid(True, alpha=0.3)

# Performance ranking
plt.subplot(2, 2, 3)
sorted_models = sorted(final_results.items(), key=lambda x: x[1]['test_rmse'])
model_names = [m[0] for m in sorted_models]
test_rmse_sorted = [m[1]['test_rmse'] for m in sorted_models]

plt.barh(model_names, test_rmse_sorted, alpha=0.7)
plt.xlabel('Test RMSE')
plt.title('Models Ranked by Test Performance')
plt.grid(True, alpha=0.3)

# Performance vs generalization trade-off
plt.subplot(2, 2, 4)
test_rmse_all = [final_results[m]['test_rmse'] for m in models]
gen_gaps = [final_results[m]['generalization_gap'] for m in models]
colors = ['green' if gap < 0.05 else 'orange' if gap < 0.1 else 'red' for gap in gen_gaps]

plt.scatter(test_rmse_all, gen_gaps, c=colors, s=100, alpha=0.7)
for i, model in enumerate(models):
    plt.annotate(model, (test_rmse_all[i], gen_gaps[i]), 
                xytext=(5, 5), textcoords='offset points', fontsize=9)
plt.xlabel('Test RMSE')
plt.ylabel('Generalization Gap')
plt.title('Performance vs Generalization Trade-off')
plt.axhline(y=0.05, color='orange', linestyle='--', alpha=0.7)
plt.axhline(y=0.1, color='red', linestyle='--', alpha=0.7)
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Model Recommendations and Analysis

In [ ]:
print("🎯 MODEL RECOMMENDATIONS & ANALYSIS:")
print("=" * 50)

# Find best model overall
best_model = min(final_results.items(), key=lambda x: x[1]['test_rmse'])
print(f"\n🏆 Best performing model: {best_model[0]}")
print(f"   Test RMSE: {best_model[1]['test_rmse']:.4f}")
print(f"   Generalization Gap: {best_model[1]['generalization_gap']:.4f}")

# Analyze overfitting for each model
print(f"\n🔍 Overfitting Analysis:")
for model_name, results in final_results.items():
    gap = results['generalization_gap']
    
    if gap < 0.01:
        status = "✅ Excellent generalization"
    elif gap < 0.05:
        status = "👍 Good generalization"
    elif gap < 0.1:
        status = "⚡ Moderate overfitting"
    elif gap < 0.2:
        status = "⚠️  High overfitting"
    else:
        status = "🚨 Severe overfitting"
    
    print(f"   {model_name}: {status} (gap: {gap:.4f})")

# Performance recommendations
print(f"\n💡 Recommendations:")
sorted_models = sorted(final_results.items(), key=lambda x: x[1]['test_rmse'])

for i, (model_name, results) in enumerate(sorted_models):
    rank = i + 1
    test_rmse = results['test_rmse']
    gap = results['generalization_gap']
    
    if rank == 1:
        print(f"   🥇 Use {model_name} for best accuracy (RMSE: {test_rmse:.4f})")
        if gap > 0.1:
            print(f"      ⚠️  Consider regularization to reduce overfitting")
    elif gap < 0.05:  # Low overfitting
        print(f"   👍 {model_name} is well-generalized (RMSE: {test_rmse:.4f}, gap: {gap:.4f})")
    elif gap > 0.15:  # High overfitting
        print(f"   ⚠️  Avoid {model_name} due to overfitting (gap: {gap:.4f})")

print(f"\n📈 Next Steps:")
print(f"   1. Use the best model for production recommendations")
print(f"   2. Monitor performance on new data")
print(f"   3. Consider ensemble methods combining multiple models")
print(f"   4. Implement online learning for continuous improvement")

## 9. Save Models and Results

In [ ]:
# Save the best models
trainer.save_models(models_dir="../models")

print("💾 Models saved successfully!")
print("\n🎉 Hyperparameter search and overfitting analysis completed!")
print("\nKey achievements:")
print("✅ Proper train/validation/test splits implemented")
print("✅ Random hyperparameter search with early stopping")
print("✅ Comprehensive overfitting detection and analysis")
print("✅ Best models identified and saved")
print("✅ Detailed performance visualizations created")

## Summary

This notebook demonstrated the enhanced model training pipeline with:

### Key Features:
1. **Proper Data Splitting**: Train (60%), Validation (20%), Test (20%)
2. **Random Hyperparameter Search**: Efficient exploration of parameter space
3. **Overfitting Detection**: Real-time monitoring of generalization gap
4. **Early Stopping**: Prevents wasteful computation
5. **Comprehensive Analysis**: Detailed reports and visualizations

### Benefits:
- **Robust Model Selection**: Unbiased evaluation on held-out test set
- **Overfitting Prevention**: Early detection and prevention of overfitting
- **Efficient Search**: Random search often outperforms grid search
- **Actionable Insights**: Clear recommendations for model deployment

### Next Steps:
- Scale to larger datasets for better generalization
- Implement ensemble methods
- Add more sophisticated regularization techniques
- Deploy best models in production environment